# Experiment 4: Logistic Regression — Binary and Multinomial Classification

**Dataset:** PlacementPredict  |  **Course Outcome:** CO2

This notebook follows `Logistic_Regression.pdf`. It uses the same dataset for two different prediction tasks:

- **Binary:** `PlacementStatus` (`Placed` / `NotPlaced`)
- **Multinomial:** `CGPA_Tier` (`Low` / `Medium` / `High`)

Run every cell in order with the **Python 3** kernel.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# Locate the complete 12-column PlacementPredict dataset.
csv_candidates = [
    Path('placement_predict_50k.csv'),
    Path('../placement_predict_50k.csv'),
    Path('/Users/laliteja/Documents/GitHub/Machine-Learning/Practicals/Gradient Descent in Practice/placement_predict_50k.csv'),
]
csv_path = next((path for path in csv_candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError('Copy the complete placement_predict_50k.csv into exp4 and run again.')

placement = pd.read_csv(csv_path)
required_columns = {'StudentID', 'Stream', 'Gender', 'CGPA_Tier', 'PlacementStatus'}
missing = required_columns - set(placement.columns)
if missing:
    raise ValueError(f'Use the complete 12-column dataset. Missing columns: {sorted(missing)}')

print('Dataset shape:', placement.shape)
print('\nPlacement status distribution:')
print(placement['PlacementStatus'].value_counts(normalize=True).round(3))
print('\nCGPA tier distribution:')
print(placement['CGPA_Tier'].value_counts(normalize=True).round(3))
placement.head()

## Preprocessing pipeline

Numeric features are standardised; text features are one-hot encoded. `StudentID` is excluded because it is only an identifier.

In [ ]:
numeric = ['CGPA', 'Internships', 'Projects', 'AptitudeScore', 'SoftSkillsRating', 'Backlogs']
categorical = ['Stream', 'Gender', 'PlacementTraining']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
])

X = placement[numeric + categorical]
y = (placement['PlacementStatus'] == 'Placed').astype(int)
print('Binary baseline (always predict Placed):', round(y.mean(), 4))

## Part A — Binary logistic regression: placement outcome

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

binary_model = Pipeline([
    ('prep', preprocessor),
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
])
binary_model.fit(X_train, y_train)

binary_predictions = binary_model.predict(X_val)
print('Baseline (always say Placed):', round(y_val.mean(), 4))
print('Model accuracy:', round(accuracy_score(y_val, binary_predictions), 4))
print('Coefficient shape:', binary_model.named_steps['lr'].coef_.shape)
print('\n', classification_report(y_val, binary_predictions, target_names=['NotPlaced', 'Placed']))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val, binary_predictions, display_labels=['NotPlaced', 'Placed'])
plt.title('Binary Logistic Regression: Placement Status')
plt.show()

feature_names = binary_model.named_steps['prep'].get_feature_names_out()
coefficients = binary_model.named_steps['lr'].coef_[0]
pd.Series(coefficients, index=feature_names).sort_values().plot.barh(figsize=(9, 5))
plt.xlabel('Coefficient')
plt.title('What raises and lowers the chance of placement')
plt.show()

## Part B — Multinomial logistic regression: CGPA tier

`CGPA` is deliberately removed before predicting `CGPA_Tier`. The tier is calculated from CGPA, so keeping it would leak the answer into the input.

In [ ]:
tier_numeric = [column for column in numeric if column != 'CGPA']
X2 = placement[tier_numeric + categorical]
y2 = placement['CGPA_Tier']

X2_train, X2_val, y2_train, y2_val = train_test_split(
    X2, y2, test_size=0.20, random_state=42, stratify=y2
)

tier_preprocessor = ColumnTransformer([
    ('num', StandardScaler(), tier_numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
])
tier_model = Pipeline([
    ('prep', tier_preprocessor),
    # scikit-learn automatically selects multinomial classification for 3 classes.
    ('lr', LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)),
])
tier_model.fit(X2_train, y2_train)

tier_predictions = tier_model.predict(X2_val)
print('Baseline (most common tier):', round(y2.value_counts(normalize=True).max(), 4))
print('Model accuracy:', round(accuracy_score(y2_val, tier_predictions), 4))
print('Classes:', list(tier_model.named_steps['lr'].classes_))
print('coef_ shape:', tier_model.named_steps['lr'].coef_.shape)
print('\n', classification_report(y2_val, tier_predictions))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y2_val, tier_predictions, display_labels=tier_model.named_steps['lr'].classes_)
plt.title('Multinomial Logistic Regression: CGPA Tier')
plt.show()

## Deliverable — compare regularisation strengths

Small `C` means stronger regularisation. Compare validation accuracy and the full per-class report, not accuracy alone.

In [ ]:
def evaluate_classifier(model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    val_predictions = model.predict(X_val)
    val_acc = accuracy_score(y_val, val_predictions)
    print('Train accuracy:', round(train_acc, 4))
    print('Validation accuracy:', round(val_acc, 4))
    print()
    print(classification_report(y_val, val_predictions, target_names=['NotPlaced', 'Placed']))
    return val_acc

models = {
    'C = 0.01 (strong penalty)': LogisticRegression(C=0.01, max_iter=1000, random_state=42),
    'C = 1 (default)': LogisticRegression(C=1, max_iter=1000, random_state=42),
    'C = 100 (weak penalty)': LogisticRegression(C=100, max_iter=1000, random_state=42),
}

scores = {}
for name, model in models.items():
    print('=' * 55)
    print(name)
    print('=' * 55)
    pipeline = Pipeline([('prep', preprocessor), ('lr', model)])
    scores[name] = evaluate_classifier(pipeline, X_train, y_train, X_val, y_val)

print('\nValidation accuracy comparison:')
print(pd.Series(scores).round(4))

## Result

Record your achieved binary and multinomial validation accuracy, each baseline, the binary coefficient interpretation, and the per-class precision/recall/F1 scores. In particular, explain why overall accuracy can hide weaker performance for `NotPlaced`.